# Lab 24-02 · Cursors, Statistical Eyes, Jitter Fits and CDR Tracking

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import eye


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Cursors

*Predict* $h_{-1}$, $h_0$, $h_1$ and $h_2$ for two equal poles with τ = 0.4 UI, then compare the analytic pulse with ngspice.

In [2]:
tp, h = eye.cursors(0.4)
print(f"peak at {tp:.4f} UI (closed form {1/(1-np.exp(-2.5)):.4f} UI)")
sim = ngspice("pulse.cir")
for k, name in ((-1, "hm1"), (0, "h0"), (1, "h1"), (2, "h2")):
    print(f"h{k:+d}: model {h[k]:.4f}   ngspice {sim[name]:.4f}")
isi = sum(abs(x) for k, x in h.items() if k)
print(f"ISI sum {isi:.4f}; NRZ eye {eye.peak_distortion_eye(h):.4f}; PAM4 eye {eye.peak_distortion_eye(h, 4):.4f}")
t = np.linspace(-0.5, 6, 1000)
plt.figure()
plt.plot(t, eye.pulse(t, 0.4), label="pulse response")
ks = np.arange(-1, 6)
plt.stem(tp + ks, [h[k] for k in ks], linefmt="C1-", markerfmt="C1o", basefmt=" ", label="cursors")
plt.xlabel("time (UI)"); plt.ylabel("amplitude"); plt.legend(); plt.grid(True)
plt.show()

peak at 1.0894 UI (closed form 1.0894 UI)
h-1: model 0.0216   ngspice 0.0216
h+0: model 0.7340   ngspice 0.7340
h+1: model 0.2109   ngspice 0.2109
h+2: model 0.0297   ngspice 0.0297
ISI sum 0.2660; NRZ eye 0.4680; PAM4 eye -0.0213


## Part 2 · Statistical against peak-distortion eye

NRZ at 32 GBd, 400 mV swing, 8 mV rms noise. *Predict* whether the statistical eye at 10⁻¹² is much taller or only slightly taller than the worst case less $2 \times 7.03\sigma$.

In [3]:
sig = 8 / 200            # noise in units of half the swing
span = isi + 1e-3
dv = 1e-5
grid = np.arange(-span, span + dv / 2, dv)
grid -= grid[np.argmin(np.abs(grid))]
pmf = eye.isi_pdf(h, grid)
f = lambda vth: np.log10(np.sum(pmf * eye.q((h[0] + grid - vth) / sig))) + 12
v1 = optimize.brentq(f, 0, h[0])
stat_h = 2 * v1 * 200
worst_h = (2 * (h[0] - isi) - 2 * float(eye.q_inv(1e-12)) * sig) * 200
print(f"statistical eye height at 1e-12: {stat_h:.1f} mV; worst case less noise: {worst_h:.1f} mV")
ber_c = eye.stat_ber_nrz(h, sig)
bound = float(eye.q((h[0] - isi) / sig))
print(f"BER at the centre: statistical {ber_c:.2e}, worst-case bound {bound:.2e} (ratio {bound/ber_c:.0f})")

statistical eye height at 1e-12: 80.6 mV; worst case less noise: 74.7 mV
BER at the centre: statistical 3.95e-33, worst-case bound 6.28e-32 (ratio 16)


## Part 3 · Fitting sinusoidal jitter

0.4 ps rms RJ plus 3 ps peak-to-peak sinusoidal PJ. *Predict* whether the dual-Dirac DJ(δδ) is above or below 3 ps.

In [4]:
def wall(b):
    return optimize.brentq(lambda x: np.log(eye.sj_rj_tail(x, 3.0, 0.4)[0]) - np.log(b), 0, 10)

x6, x9, x12 = wall(1e-6), wall(1e-9), wall(1e-12)
for (xa, ba), (xb, bb) in (((x9, 1e-9), (x12, 1e-12)), ((x6, 1e-6), (x9, 1e-9))):
    s, dj = eye.qscale_fit(xa, ba, xb, bb)
    print(f"fit through {ba:.0e} and {bb:.0e}: sigma {s:.3f} ps, DJ(dd) {dj:.2f} ps")
x = np.linspace(1.0, 4.5, 300)
plt.figure()
plt.plot(x, eye.q_inv(eye.sj_rj_tail(x, 3.0, 0.4)), label="sinusoidal PJ + RJ (exact)")
s, dj = eye.qscale_fit(x9, 1e-9, x12, 1e-12)
plt.plot(x, (x - dj / 2) / s, "--", label="dual-Dirac fit (1e-9, 1e-12)")
plt.xlabel("distance beyond the ideal crossing (ps)"); plt.ylabel("Q (tail probability)")
plt.legend(); plt.grid(True)
plt.show()

fit through 1e-09 and 1e-12: sigma 0.419 ps, DJ(dd) 2.44 ps
fit through 1e-06 and 1e-09: sigma 0.427 ps, DJ(dd) 2.35 ps


## Part 4 · CDR tracking

A first-order loop with $f_c$ = 4 MHz tracks unit sinusoidal jitter. *Predict* the tracking-error ratio at 100 kHz, 4 MHz and 40 MHz.

In [5]:
for fj in (1e5, 4e6, 4e7):
    print(f"{fj/1e6:6.1f} MHz: simulated {eye.cdr_tracking_sim(fj, 4e6):.4f}, "
          f"|1 - H| = {float(eye.tracking_error_ratio(fj, 4e6)):.4f}")
f = np.logspace(4, 9, 300)
plt.figure()
plt.loglog(f / 1e6, eye.jitter_tolerance(f, 4e6, 0.3), label="first-order CDR, M = 0.3 UI")
plt.xlabel("jitter frequency (MHz)"); plt.ylabel("tolerable jitter (UI p-p)"); plt.legend(); plt.grid(True, which="both")
plt.show()

   0.1 MHz: simulated 0.0250, |1 - H| = 0.0250


   4.0 MHz: simulated 0.7071, |1 - H| = 0.7071


  40.0 MHz: simulated 0.9950, |1 - H| = 0.9950
